In [ ]:
%sql

--Cria catalog e schema
CREATE CATALOG IF NOT EXISTS lh_nautical;
CREATE SCHEMA IF NOT EXISTS lh_nautical.bronze;

In [ ]:

from databricks.sdk.runtime import dbutils, spark 

# Dados da conexão - Ideal é estarem numa variável de ambiente e serem puxadas de lá
db_user = dbutils.secrets.get(scope="sqlserver", key="username")
db_password = dbutils.secrets.get(scope="sqlserver", key="password")
db_host = dbutils.secrets.get(scope="sqlserver", key="host")
db_port = dbutils.secrets.get(scope="sqlserver", key="port")
db_database = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{db_host}:{db_port};databaseName={db_database}"


# Lista das tabelas que vão ser extraídas - Geralmente é obtido através de outro banco de dados que chama de ControlDB, mas aqui no nosso projeto não vejo problema de apenas colocarmos eles num JSON dentro do databricks
tabelas_ecommerce = [
 "customers",
 "orders",
 "order_items",
 "products",
 "product_variants",
 "brands",
 "categories",
 "returns",
 "return_items",
 "stock_levels",
 "stock_movements",
 "suppliers",
 "employees",
 "locations",
 "addresses"
 ]


# Loop para processar cada tabela baseando-se nos dados da conexão declarados anteriormente
for tabela in tabelas_ecommerce:
    print(f"Iniciando extração da tabela: {tabela}...")
    
    jdbc_options = {
    "url":      jdbc_url,
    "dbtable":  tabela,
    "user":     db_user,
    "password": db_password,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "encrypt":  "true",
    "trustServerCertificate": "true",
    "loginTimeout": "30",
    "fetchsize": "1000",
    }
    
    
    # Lendo a tabela atual do SQL Server
    df_temp = (
        spark.read 
        .format("jdbc") 
        .options(**jdbc_options) 
        .load()
    )
    
    output_table = f"lh_nautical.bronze.{tabela}"
    
    df_temp.write.format("delta").mode("overwrite").saveAsTable(output_table)
    print(f"Tabela {output_table} salva!")
    
    #print(f"  {df_temp.count()} linhas carregadas.")
    #display(df_temp)  #Printa na tela as tabelas   